In [23]:
import numpy as np
import pandas as pd
import re
import os
root_dir = os.path.dirname(os.getcwd())  # Root directory of the project
root_dir

'c:\\Users\\Admin\\Documents\\GitHub\\explain-error'

In [24]:
df = pd.read_csv(os.path.join(root_dir,"data","chexpert","train_visualCheXbert.csv.crdownload"))
df

,Path,Sex,Age,Frontal/Lateral,AP/PA,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding
0,CheXpert-v1.0/train/patient00001/study1/view1_...,Female,68,Frontal,AP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,CheXpert-v1.0/train/patient00002/study2/view1_...,Female,87,Frontal,AP,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,CheXpert-v1.0/train/patient00002/study1/view1_...,Female,83,Frontal,AP,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
3,CheXpert-v1.0/train/patient00002/study1/view2_...,Female,83,Lateral,NaN,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
4,CheXpert-v1.0/train/patient00003/study1/view1_...,Male,41,Frontal,AP,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223409,CheXpert-v1.0/train/patient64537/study2/view1_...,Male,59,Frontal,AP,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0
223410,CheXpert-v1.0/train/patient64537/study1/view1_...,Male,59,Frontal,AP,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0
223411,CheXpert-v1.0/train/patient64538/study1/view1_...,Female,0,Frontal,AP,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
223412,CheXpert-v1.0/train/patient64539/study1/view1_...,Female,0,Frontal,AP,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [25]:
df=df[["Path", "Sex","Age","No Finding"]]
df

,Path,Sex,Age,No Finding
0,CheXpert-v1.0/train/patient00001/study1/view1_...,Female,68,0.0
1,CheXpert-v1.0/train/patient00002/study2/view1_...,Female,87,0.0
2,CheXpert-v1.0/train/patient00002/study1/view1_...,Female,83,0.0
3,CheXpert-v1.0/train/patient00002/study1/view2_...,Female,83,0.0
4,CheXpert-v1.0/train/patient00003/study1/view1_...,Male,41,0.0
...,...,...,...,...
223409,CheXpert-v1.0/train/patient64537/study2/view1_...,Male,59,0.0
223410,CheXpert-v1.0/train/patient64537/study1/view1_...,Male,59,0.0
223411,CheXpert-v1.0/train/patient64538/study1/view1_...,Female,0,0.0
223412,CheXpert-v1.0/train/patient64539/study1/view1_...,Female,0,0.0


In [26]:

# Create a new column that groups ages into 20-year intervals
bins = [0,20,40,60,80,200]  # Define the bins (from 0 to 100, step by 20)
labels = [f'{i}-{i+20}' for i in bins[:-2]]  # Create labels like '0-19', '20-39', etc.
labels.append('80+')
df['age_decile'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)
df


C:\Users\Admin\AppData\Local\Temp\ipykernel_16732\2945828636.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['age_decile'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)


,Path,Sex,Age,No Finding,age_decile
0,CheXpert-v1.0/train/patient00001/study1/view1_...,Female,68,0.0,60-80
1,CheXpert-v1.0/train/patient00002/study2/view1_...,Female,87,0.0,80+
2,CheXpert-v1.0/train/patient00002/study1/view1_...,Female,83,0.0,80+
3,CheXpert-v1.0/train/patient00002/study1/view2_...,Female,83,0.0,80+
4,CheXpert-v1.0/train/patient00003/study1/view1_...,Male,41,0.0,40-60
...,...,...,...,...,...
223409,CheXpert-v1.0/train/patient64537/study2/view1_...,Male,59,0.0,40-60
223410,CheXpert-v1.0/train/patient64537/study1/view1_...,Male,59,0.0,40-60
223411,CheXpert-v1.0/train/patient64538/study1/view1_...,Female,0,0.0,0-20
223412,CheXpert-v1.0/train/patient64539/study1/view1_...,Female,0,0.0,0-20


In [27]:
df["age_decile"].unique()

['60-80', '80+', '40-60', '20-40', '0-20']
Categories (5, object): ['0-20' < '20-40' < '40-60' < '60-80' < '80+']

In [28]:
df = df.rename(columns={'Sex': 'gender'})
df['gender'] = df['gender'].replace({'Female': 'F', 'Male': 'M'})
df

,Path,gender,Age,No Finding,age_decile
0,CheXpert-v1.0/train/patient00001/study1/view1_...,F,68,0.0,60-80
1,CheXpert-v1.0/train/patient00002/study2/view1_...,F,87,0.0,80+
2,CheXpert-v1.0/train/patient00002/study1/view1_...,F,83,0.0,80+
3,CheXpert-v1.0/train/patient00002/study1/view2_...,F,83,0.0,80+
4,CheXpert-v1.0/train/patient00003/study1/view1_...,M,41,0.0,40-60
...,...,...,...,...,...
223409,CheXpert-v1.0/train/patient64537/study2/view1_...,M,59,0.0,40-60
223410,CheXpert-v1.0/train/patient64537/study1/view1_...,M,59,0.0,40-60
223411,CheXpert-v1.0/train/patient64538/study1/view1_...,F,0,0.0,0-20
223412,CheXpert-v1.0/train/patient64539/study1/view1_...,F,0,0.0,0-20


In [29]:
df['No Finding'] = df['No Finding'].fillna(0.0)
df

,Path,gender,Age,No Finding,age_decile
0,CheXpert-v1.0/train/patient00001/study1/view1_...,F,68,0.0,60-80
1,CheXpert-v1.0/train/patient00002/study2/view1_...,F,87,0.0,80+
2,CheXpert-v1.0/train/patient00002/study1/view1_...,F,83,0.0,80+
3,CheXpert-v1.0/train/patient00002/study1/view2_...,F,83,0.0,80+
4,CheXpert-v1.0/train/patient00003/study1/view1_...,M,41,0.0,40-60
...,...,...,...,...,...
223409,CheXpert-v1.0/train/patient64537/study2/view1_...,M,59,0.0,40-60
223410,CheXpert-v1.0/train/patient64537/study1/view1_...,M,59,0.0,40-60
223411,CheXpert-v1.0/train/patient64538/study1/view1_...,F,0,0.0,0-20
223412,CheXpert-v1.0/train/patient64539/study1/view1_...,F,0,0.0,0-20


In [30]:
df = df.loc[df['age_decile'] != '0-20']

In [31]:
df=df[["Path", "gender","age_decile","No Finding"]]
df["PATIENT"] = df["Path"].str.extract(r'(patient\d{5})')
df["PATIENT"]

0         patient00001
1         patient00002
2         patient00002
3         patient00002
4         patient00003
              ...     
223406    patient64535
223407    patient64536
223408    patient64536
223409    patient64537
223410    patient64537
Name: PATIENT, Length: 221478, dtype: object

In [32]:
df2 = pd.read_excel(os.path.join(root_dir,"data","chexpert","CHEXPERT DEMO.xlsx"))
df2

,PATIENT,GENDER,AGE_AT_CXR,PRIMARY_RACE,ETHNICITY
0,patient24428,Male,61,White,Non-Hispanic/Non-Latino
1,patient48289,Female,39,Other,Hispanic/Latino
2,patient33856,Female,81,White,Non-Hispanic/Non-Latino
3,patient41673,Female,42,Unknown,Unknown
4,patient48493,Male,71,White,Non-Hispanic/Non-Latino
...,...,...,...,...,...
65396,patient65702,Male,1,Other,Hispanic/Latino
65397,patient04979,Female,27,Other,Hispanic/Latino
65398,patient11445,Female,29,Unknown,Unknown
65399,patient23235,Female,41,"Other, Hispanic",Hispanic/Latino


In [33]:
df2=df2.fillna("")
df2["PRIMARY_RACE"].unique()

array(['White', 'Other', 'Unknown', 'White, non-Hispanic', 'Asian', '',
       'Black or African American', 'Black, non-Hispanic',
       'Other, Hispanic', 'Race and Ethnicity Unknown',
       'Asian, non-Hispanic', 'Pacific Islander, non-Hispanic',
       'Native Hawaiian or Other Pacific Islander', 'Other, non-Hispanic',
       'Patient Refused', 'White, Hispanic', 'Black, Hispanic',
       'Asian, Hispanic', 'American Indian or Alaska Native',
       'Native American, Hispanic', 'Native American, non-Hispanic',
       'Pacific Islander, Hispanic', 'Asian - Historical Conv',
       'White or Caucasian'], dtype=object)

In [34]:
df2["ETHNICITY"].unique()


array(['Non-Hispanic/Non-Latino', 'Hispanic/Latino', 'Unknown', '',
       'Patient Refused', 'Hispanic', 'Not Hispanic'], dtype=object)

In [35]:
# Function to categorize the data into the desired groups
def categorize(row):
    # Check for Hispanic/Latino first in column B, if not found then check column A
    if 'Hispanic/Latino'==str(row["ETHNICITY"]) or 'Hispanic'==str(row["ETHNICITY"]) or ' Hispanic' in str(row["PRIMARY_RACE"]):
        if 'Black' in str(row["PRIMARY_RACE"]):
            return 'Black'
        elif 'Asian' in str(row["PRIMARY_RACE"]):
            return 'Asian'
        else:
            return 'Hispanic/Latino'
    else:
        if 'Black' in str(row["PRIMARY_RACE"]):
            return 'Black'
        elif 'Asian' in str(row["PRIMARY_RACE"]):
            return 'Asian'
        elif 'White' in str(row["PRIMARY_RACE"]):
            return 'White'
        elif 'Patient' in str(row["PRIMARY_RACE"]) or 'Uknown' in str(row["PRIMARY_RACE"]) or ''==str(row["PRIMARY_RACE"]):
            return 'Unknown'
        else:
            return 'Other'

df2['grouped_race'] = df2[['PRIMARY_RACE', 'ETHNICITY']].apply(categorize, axis=1)
df2['grouped_race'].value_counts(normalize=True)

grouped_race
White              0.539762
Other              0.166297
Hispanic/Latino    0.127460
Asian              0.107965
Black              0.048119
Unknown            0.010397
Name: proportion, dtype: float64

In [36]:
df2=df2[["PATIENT", "grouped_race"]]
df2

,PATIENT,grouped_race
0,patient24428,White
1,patient48289,Hispanic/Latino
2,patient33856,White
3,patient41673,Other
4,patient48493,White
...,...,...
65396,patient65702,Hispanic/Latino
65397,patient04979,Hispanic/Latino
65398,patient11445,Other
65399,patient23235,Hispanic/Latino


In [37]:
df2 = df2.loc[df2['grouped_race'] != 'Unknown']
df2 = df2.loc[df2['grouped_race'] != 'Asian']
df = pd.merge(df, df2, on='PATIENT', how='inner')
df

,Path,gender,age_decile,No Finding,PATIENT,grouped_race
0,CheXpert-v1.0/train/patient00001/study1/view1_...,F,60-80,0.0,patient00001,Other
1,CheXpert-v1.0/train/patient00002/study2/view1_...,F,80+,0.0,patient00002,White
2,CheXpert-v1.0/train/patient00002/study1/view1_...,F,80+,0.0,patient00002,White
3,CheXpert-v1.0/train/patient00002/study1/view2_...,F,80+,0.0,patient00002,White
4,CheXpert-v1.0/train/patient00003/study1/view1_...,M,40-60,0.0,patient00003,White
...,...,...,...,...,...,...
195682,CheXpert-v1.0/train/patient64535/study1/view1_...,M,60-80,1.0,patient64535,Black
195683,CheXpert-v1.0/train/patient64536/study2/view1_...,F,60-80,0.0,patient64536,Hispanic/Latino
195684,CheXpert-v1.0/train/patient64536/study1/view1_...,F,60-80,0.0,patient64536,Hispanic/Latino
195685,CheXpert-v1.0/train/patient64537/study2/view1_...,M,40-60,0.0,patient64537,Black


In [38]:
df["insurance"]=[np.nan for i in range(len(df))]
df

,Path,gender,age_decile,No Finding,PATIENT,grouped_race,insurance
0,CheXpert-v1.0/train/patient00001/study1/view1_...,F,60-80,0.0,patient00001,Other,NaN
1,CheXpert-v1.0/train/patient00002/study2/view1_...,F,80+,0.0,patient00002,White,NaN
2,CheXpert-v1.0/train/patient00002/study1/view1_...,F,80+,0.0,patient00002,White,NaN
3,CheXpert-v1.0/train/patient00002/study1/view2_...,F,80+,0.0,patient00002,White,NaN
4,CheXpert-v1.0/train/patient00003/study1/view1_...,M,40-60,0.0,patient00003,White,NaN
...,...,...,...,...,...,...,...
195682,CheXpert-v1.0/train/patient64535/study1/view1_...,M,60-80,1.0,patient64535,Black,NaN
195683,CheXpert-v1.0/train/patient64536/study2/view1_...,F,60-80,0.0,patient64536,Hispanic/Latino,NaN
195684,CheXpert-v1.0/train/patient64536/study1/view1_...,F,60-80,0.0,patient64536,Hispanic/Latino,NaN
195685,CheXpert-v1.0/train/patient64537/study2/view1_...,M,40-60,0.0,patient64537,Black,NaN


In [39]:
df.to_csv(os.path.join(root_dir,"data","chexpert","metadata.csv"))

In [17]:
a=np.load(os.path.join(root_dir,"data","chexpert","train.npz"))

In [18]:
lst=a.files
paths=[item for item in lst]
embed=[a[item] for item in lst]

In [19]:
len(paths)

223414

In [20]:

df2 = pd.DataFrame({
    'Path': paths,
    'embedding': embed
})
df2.head(10)

,Path,embedding
0,CheXpert-v1.0/train/patient00001/study1/view1_...,"[-0.6499777436256409, -2.122708320617676, 0.84..."
1,CheXpert-v1.0/train/patient00002/study1/view1_...,"[0.45629817247390747, -1.0862308740615845, 0.7..."
2,CheXpert-v1.0/train/patient00002/study1/view2_...,"[-0.022207040339708328, -1.7003875970840454, 0..."
3,CheXpert-v1.0/train/patient00002/study2/view1_...,"[0.17246507108211517, -1.5009434223175049, 0.5..."
4,CheXpert-v1.0/train/patient00003/study1/view1_...,"[-0.22483156621456146, -1.8942707777023315, 0...."
5,CheXpert-v1.0/train/patient00004/study1/view1_...,"[-0.0995982438325882, -2.2466962337493896, 1.1..."
6,CheXpert-v1.0/train/patient00004/study1/view2_...,"[-0.9989555478096008, -2.05975604057312, 0.260..."
7,CheXpert-v1.0/train/patient00005/study1/view1_...,"[-0.04464299976825714, -2.1811063289642334, 1...."
8,CheXpert-v1.0/train/patient00005/study1/view2_...,"[-0.8779672384262085, -2.1962430477142334, 1.6..."
9,CheXpert-v1.0/train/patient00005/study2/view1_...,"[-0.5502946376800537, -1.9120078086853027, 0.2..."


In [21]:
import pickle
with open(os.path.join(root_dir,"data","chexpert","embed.pkl"),"wb") as f:
    pickle.dump(df2,f)

In [22]:
df["age_decile"].unique()

['60-80', '80+', '40-60', '20-40', NaN]
Categories (5, object): ['0-20' < '20-40' < '40-60' < '60-80' < '80+']